# 🔄 Fluxuri de lucru de bază pentru agenți cu Microsoft Foundry (Python)

## 📋 Tutorial de orchestrare a fluxului de lucru

Acest notebook prezintă puternicele capabilități ale **Workflow Builder** din Microsoft Agent Framework. Învățați cum să creați fluxuri de lucru sofisticate, cu mai mulți pași, care pot gestiona procese de afaceri complexe și pot coordona multiple operații AI fără întreruperi.

> **Notă de migrare:** Acest exemplu făcea referire anterior la Modelele GitHub. Modelele GitHub sunt retrase (se vor elimina în iulie 2026), așadar acum se utilizează **Microsoft Foundry** prin `FoundryChatClient`, care țintește Azure OpenAI **Responses API**.

## 🎯 Obiectivele de învățare

### 🏗️ **Arhitectura fluxului de lucru**
- **Workflow Builder**: Proiectați și orchestrați procese complexe cu mai mulți pași
- **Execuție bazată pe evenimente**: Gestionați evenimentele fluxului de lucru și tranzițiile de stare
- **Design vizual al fluxului de lucru**: Creați și vizualizați structurile fluxului de lucru
- **Integrare Microsoft Foundry**: Valorificați modelele AI în contextul fluxurilor de lucru

### 🔄 **Orchestrarea proceselor**
- **Operațiuni secvențiale**: Conectați sarcinile agentului în ordine logică
- **Logică condițională**: Implementați puncte decizionale și fluxuri ramificate
- **Gestionarea erorilor**: Recuperare robustă a erorilor și reziliență a fluxului de lucru
- **Gestionarea stării**: Urmăriți și gestionați starea de execuție a fluxului de lucru

### 📊 **Tipare de flux de lucru pentru întreprindere**
- **Automatizarea proceselor de afaceri**: Automatizați fluxuri de lucru organizaționale complexe
- **Coordonarea multi-agent**: Coordonați mai mulți agenți specializați
- **Execuție scalabilă**: Proiectați fluxuri de lucru pentru operațiuni la scară de întreprindere
- **Monitorizare și observabilitate**: Urmăriți performanța și rezultatele fluxurilor de lucru

## ⚙️ Cerințe prealabile și configurare

### 📦 **Dependențe necesare**

Instalați Agent Framework cu capabilități de flux de lucru:

```bash
pip install agent-framework -U
```

### 🔑 **Configurare Microsoft Foundry**

Conectați-vă cu Azure CLI (`az login`) pentru ca `AzureCliCredential` să poată autentifica, apoi setați detaliile proiectului Microsoft Foundry.

**Configurare mediu (.env file):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

### 🏢 **Cazuri de utilizare în întreprindere**

**Exemple de procese de afaceri:**
- **Integrarea clienților**: Fluxuri de lucru multi-pas pentru verificare și configurare
- **Flux de conținut**: Creare, revizuire și publicare automată a conținutului
- **Procesarea datelor**: Fluxuri ETL cu transformări bazate pe AI
- **Asigurarea calității**: Procese automate de testare și validare

**Beneficiile fluxurilor de lucru:**
- 🎯 **Fiabilitate**: Execuție deterministă cu recuperare la erori
- 📈 **Scalabilitate**: Gestionați automatizarea cu volum mare
- 🔍 **Observabilitate**: Trasabilitate completă și monitorizare
- 🔧 **Mentenenabilitate**: Design vizual și componente modulare

## 🎨 Tipare de design al fluxurilor de lucru

### Structura de bază a fluxului de lucru
```mermaid
graph TD
    A[Start] --> B[Sarcina Agentului 1]
    B --> C{Punct de Decizie}
    C -->|Succes| D[Sarcina Agentului 2]
    C -->|Eșec| E[Gestionar de Erori]
    D --> F[Sfârșit]
    E --> F
```

**Componente cheie:**
- **WorkflowBuilder**: Motorul principal de orchestrare
- **WorkflowEvent**: Gestionarea și comunicarea evenimentelor
- **WorkflowViz**: Reprezentare vizuală și depanare a fluxului de lucru

Să construim primul tău flux de lucru inteligent! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
